# 05 — Negative Reason Classification

**Sentimentanalys av flygbolagstweets — steg 5/6**

I Biometric Access Terminal-projektet var `05_age_gender_estimation.ipynb` en **andra,
kompletterande** klassificeringsuppgift på samma rådata. Här gör vi samma sak för vårt
textproblem: bland de tweets som redan är **negativa**, tränar vi en ny DistilBERT-modell
att klassificera **varför** — försening, kundservice, borttappat bagage, osv. Det är samma
transfer learning-recept som i notebook 04, applicerat på en ny etikett.

För en flygbolagets kundtjänst är det här minst lika värdefullt som själva
sentimentklassificeringen: att veta att en tweet är negativ räcker inte för att veta vem som
ska hantera ärendet — orsaken avgör det.

> **Kräver internet + GPU:** kör i **Google Colab** (Runtime → Change runtime type → GPU).


## Installation & imports

In [ ]:
!pip install -q transformers tensorflow scikit-learn pandas matplotlib seaborn


In [ ]:
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, TFDistilBertModel

sns.set_style("whitegrid")
tf.random.set_seed(42)
np.random.seed(42)

os.makedirs("models", exist_ok=True)


## Data

Till skillnad från notebook 04 använder vi **inte** `data/train.csv` här — den är ett litet
urval stratifierat efter *sentiment*, vilket skulle ge för få exempel av vissa orsaker. Vi
går istället tillbaka till `data/full_clean.csv` från notebook 01 och gör ett eget,
större urval — stratifierat efter *orsak* — bara från de negativa tweetsen.


In [ ]:
full_df = pd.read_csv("data/full_clean.csv")

negative_df = full_df[
    (full_df["airline_sentiment"] == "negative") & (full_df["negativereason"].notna())
].copy()

print("Negativa tweets med angiven orsak:", negative_df.shape)
negative_df["negativereason"].value_counts()


**Vad output visar:** runt ~9 100 negativa tweets har en angiven orsak.
`value_counts()` listar alla ursprungliga orsakskategorier — de är många och väldigt ojämnt
fördelade (från **Customer Service Issue** med tusentals exempel ner till kategorier med
under 100). För många små klasser gör klassificeringen rörig och svår att utvärdera — därför
grupperar vi om dem härnäst.


## Gruppera till hanterbara klasser

Vi behåller de fem största, mest konkreta orsakerna och slår ihop resten (inklusive den
diffusa kategorin "Can't Tell") till **Other**.


In [ ]:
TOP_REASONS = [
    "Customer Service Issue",
    "Late Flight",
    "Cancelled Flight",
    "Lost Luggage",
    "Bad Flight",
]

negative_df["reason_group"] = negative_df["negativereason"].apply(
    lambda r: r if r in TOP_REASONS else "Other"
)

reason_classes = sorted(negative_df["reason_group"].unique())
reason2id = {r: i for i, r in enumerate(reason_classes)}
id2reason = {i: r for r, i in reason2id.items()}

print("Klasser:", reason_classes)
negative_df["reason_group"].value_counts()


**Vad output visar:** nu bara 6 klasser (de 5 valda + "Other"). Jämför siffrorna med
diagrammet "Vanligaste orsakerna" från notebook 02 — **Customer Service Issue** och
**Late Flight** är fortfarande klart störst. Klasserna är fortfarande ojämnt fördelade
(precis som verkligheten är), vilket vi tar hänsyn till när vi tolkar resultaten i notebook 06.


## Urval & train/val/test-delning

In [ ]:
SAMPLE_SIZE = 3000

sample, _ = train_test_split(
    negative_df, train_size=min(SAMPLE_SIZE, len(negative_df)),
    stratify=negative_df["reason_group"], random_state=42
)

train_val, reason_test_df = train_test_split(
    sample, test_size=0.15, stratify=sample["reason_group"], random_state=42
)
reason_train_df, reason_val_df = train_test_split(
    train_val, test_size=0.176, stratify=train_val["reason_group"], random_state=42
)

print("Train:", reason_train_df.shape, " Val:", reason_val_df.shape, " Test:", reason_test_df.shape)

reason_train_df.to_csv("data/reason_train.csv", index=False)
reason_val_df.to_csv("data/reason_val.csv", index=False)
reason_test_df.to_csv("data/reason_test.csv", index=False)
print("Sparade data/reason_train.csv, reason_val.csv, reason_test.csv")


**Vad output visar:** train/val/test-storlekar (ca 2100/370/450 rader) plus en
bekräftelse att de tre nya CSV-filerna sparats — dessa återanvänds av notebook 06 för
utvärdering, precis som `data/test.csv` används för sentimentmodellen.


## Basmodell, frysning & egna lager

Exakt samma recept som i notebook 04 — DistilBERT som feature extractor, fryst till att
börja med, med egna klassificeringslager ovanpå. Enda skillnaden är antalet utklasser
(6 istället för 3).


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64
N_CLASSES = len(reason_classes)

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(texts):
    enc = tokenizer(list(texts), max_length=MAX_LEN, truncation=True, padding="max_length", return_tensors="tf")
    return enc["input_ids"], enc["attention_mask"]

X_train_ids, X_train_mask = tokenize(reason_train_df["text"])
X_val_ids, X_val_mask = tokenize(reason_val_df["text"])
X_test_ids, X_test_mask = tokenize(reason_test_df["text"])

y_train = reason_train_df["reason_group"].map(reason2id).values
y_val = reason_val_df["reason_group"].map(reason2id).values
y_test = reason_test_df["reason_group"].map(reason2id).values


In [ ]:
bert_base = TFDistilBertModel.from_pretrained(MODEL_NAME)
bert_base.trainable = False

input_ids = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

bert_outputs = bert_base(input_ids, attention_mask=attention_mask)
cls_token = bert_outputs.last_hidden_state[:, 0, :]

x = tf.keras.layers.Dense(128, activation="relu")(cls_token)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(N_CLASSES, activation="softmax", name="reason_output")(x)

reason_model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output, name="negative_reason_distilbert")

reason_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
reason_model.summary()


**Vad output visar:** samma typ av lagerlista som i notebook 04, men lägg märke till
att sista lagret (`reason_output`) nu har **6** utgångar istället för 3 — en per orsakskategori.


## Träning: frusen bas följt av fine-tuning

In [ ]:
BATCH_SIZE = 32
EPOCHS_PHASE1 = 3

history1 = reason_model.fit(
    x=[X_train_ids, X_train_mask], y=y_train,
    validation_data=([X_val_ids, X_val_mask], y_val),
    epochs=EPOCHS_PHASE1, batch_size=BATCH_SIZE,
    steps_per_epoch=math.ceil(len(y_train) / BATCH_SIZE),
    validation_steps=math.ceil(len(y_val) / BATCH_SIZE),
)


**Vad output visar:** träningslogg med accuracy/loss per epok, precis som i
notebook 04. Eftersom vi nu har **6 klasser** istället för 3 är uppgiften svårare rent
statistiskt — förvänta dig därför något lägre accuracy än för sentimentmodellen, även efter
fine-tuning.


In [ ]:
bert_base.trainable = True
for layer in bert_base.distilbert.transformer.layer[:-2]:
    layer.trainable = False

reason_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_PHASE2 = 2

history2 = reason_model.fit(
    x=[X_train_ids, X_train_mask], y=y_train,
    validation_data=([X_val_ids, X_val_mask], y_val),
    epochs=EPOCHS_PHASE2, batch_size=BATCH_SIZE,
    steps_per_epoch=math.ceil(len(y_train) / BATCH_SIZE),
    validation_steps=math.ceil(len(y_val) / BATCH_SIZE),
)


**Vad output visar:** ytterligare träningsepoker efter att de två sista
transformerblocken låsts upp. `val_accuracy` bör fortsätta öka (eller åtminstone inte
försämras) — annars kan det vara värt att pröva färre fine-tuning-epoker.


In [ ]:
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(acc, label="Träning")
ax.plot(val_acc, label="Validering")
ax.axvline(EPOCHS_PHASE1 - 0.5, color="gray", linestyle="--", label="Fine-tuning startar")
ax.set_title("Accuracy per epok — orsaksklassificering")
ax.set_xlabel("Epok"); ax.legend()
plt.tight_layout()
plt.savefig("charts/reason_training_curves.png", dpi=150)
plt.show()


**Vad diagrammet visar:** samma typ av träningskurva som i notebook 04, fast för
orsaksmodellen. Använd den för att jämföra hur mycket svårare/lättare den här uppgiften är
jämfört med sentimentklassificeringen i notebook 04.


## Spara modellen för notebook 06

In [ ]:
reason_model.save("models/reason_model.keras")
tokenizer.save_pretrained("models/reason_tokenizer")

import json
with open("models/reason_classes.json", "w") as f:
    json.dump(reason_classes, f)

print("Sparade models/reason_model.keras, models/reason_tokenizer/ och models/reason_classes.json")


**Vad output visar:** en bekräftelse att modellen, tokenizern **och** listan med
klassnamn sparats. Vi sparar klassnamnen separat i en liten JSON-fil eftersom ordningen på
klasserna (0=Bad Flight, 1=Cancelled Flight, osv.) annars bara finns i minnet här — utan den
filen skulle notebook 06 inte veta vilket nummer som betyder vad.


## Sammanfattning

Vi har nu tränat en andra DistilBERT-modell med samma transfer learning-recept som i
notebook 04, men för en ny uppgift: att klassificera **orsaken** bakom en negativ tweet i
6 kategorier. Tillsammans täcker våra två modeller både *vad* kunden känner och *varför*.

**Nästa steg:** öppna `06_evaluation.ipynb` för att utvärdera båda modellerna tillsammans.
